# Chapter 1: Introduction

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch01_introduction.ipynb)


In [ ]:
# Colab does not ship these. Running this cell is a no-op if they are already present.
%pip install -q transformers


In [1]:
# Seeded before anything below runs, including the chapter's own examples.
#
# Those examples call torch.randn without a seed. This notebook is committed
# with its output stored, and a stored number that changes on every rebuild is
# noise printed as a result. Seeding here makes the whole file reproducible.
#
# Same seed as tools/claim_instances.py, which produced the slide numbers.
import torch
torch.manual_seed(20260729)
print('seeded', 20260729)

seeded 20260729


### 1.1.2 Prediction as the Common Thread

The following code example makes this abstract discussion concrete. By loading a pre-trained GPT-2 model and running a single forward pass on the prompt "The cat sat on the," we can examine the probability distribution over all $50{,}257$ tokens in GPT-2's vocabulary and inspect the five highest-ranked candidates.


In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load pre-trained GPT-2 and its tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Encode the prompt and get the model's predictions
prompt = "The cat sat on the"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

# Extract probabilities for the next token
next_token_probs = torch.softmax(logits[0, -1, :], dim=0)
top5 = torch.topk(next_token_probs, 5)

# Display the top 5 predicted next words with their probabilities
for prob, idx in zip(top5.values, top5.indices):
    token = tokenizer.decode(idx)
    print(f"  {token.strip():>12s}  {prob.item():.4f}")


         floor  0.0764
           bed  0.0653
         couch  0.0541
        ground  0.0521
          edge  0.0478


### 1.1.3 From Probabilities to Applications

A notable feature of the prediction objective is how many apparently different NLP tasks it subsumes. Text generation is the most direct application: we sample from $P(w_t \mid w_{<t})$ repeatedly, feeding each predicted token back into the context and sampling again, until we have produced a sequence of the desired length.


In [3]:
from transformers import pipeline, set_seed

# Load a pre-trained language model for text generation
generator = pipeline("text-generation", model="gpt2")
set_seed(42)  # fix the random seed for reproducibility

# Generate text by predicting one token at a time
prompt = "The future of artificial intelligence"
output = generator(prompt, max_new_tokens=50, do_sample=True, temperature=0.8)

# The model generates fluent text solely by predicting the next token repeatedly
print(output[0]["generated_text"])


The future of artificial intelligence is bound to be bright.

A new report from the Center for Computer and Information Sciences (CCIS), a nonprofit think tank that advocates for privacy and safety in artificial intelligence, is raising questions about how AI will replace human-level intelligence.


---

## Summary

This notebook demonstrated the key code examples from Chapter 1: Introduction. For the full mathematical exposition and discussion, refer to the textbook chapter.


---

## The two numbers chapter 1 asks you to see

Chapter 1 makes one claim it cannot show you on a slide and one it can only
assert. **A forward pass returns a distribution over the whole vocabulary, not
a word**, and **an n-gram table's contexts grow as the vocabulary raised to the
context length**. The frames show the code and the table. This runs both.

The cells below are **the lecture's own code**, lifted verbatim from
`tools/claim_instances.py`, which is what generated the numbers on the slides.
Their output is stored in this file, so the notebook is readable without running
anything and checkable by running it.

The GPT-2 checkpoint is **pinned to a revision**, printed by the setup cell. It
has to be: `gpt2` on its own means whatever the hub is serving today, and the
numbers below would then match the slides once and drift silently afterwards.
Nothing here samples, so there is no seed to pin; the results depend on that
checkpoint and on nothing else.

Two things to do rather than read. Change `PROMPT_CH01` and watch how much
probability mass moves out of the top five: a prompt that constrains the next
word hard gives a spiky distribution, and a vague one gives a flat one, and the
model is doing the same thing in both cases. Then print the distribution at an
**earlier** position instead of the last one. Every position carries one, which
is the fact chapter 9 turns into a training objective.

One thing worth doing on the second cell. Put your own `V_CH01` in, something
small enough to feel reasonable, and see how far you have to shrink the
vocabulary before a trillion-word corpus could fill a 5-gram table. The answer
is the whole argument of the block: the table is not merely big, it is bigger
than any amount of text, so widening the window makes the sparsity worse rather
than better.


In [4]:
# Lifted from tools/claim_instances.py, which generated every number in
# the chapter 1 slides. Do not retype these: rerun
#     python tools/build_notebook.py ch01
# and the notebook picks up whatever the module now says.
from decimal import Decimal, ROUND_HALF_UP
import torch

MODEL_CH01 = 'gpt2'
REVISION_CH01 = '607a30d783dfa663caf39e06633721c8d4cfcd7e'
PROMPT_CH01 = 'The cat sat on the'
V_CH01 = 50000
CORPUS_CH01 = 1000000000000

def tex_int(n):
    """Format an integer the way the deck does: 1{,}048{,}576."""
    return "{,}".join(reversed([str(n)[max(0, i - 3):i]
                                for i in range(len(str(n)), 0, -3)]))

def _sci_tex(x):
    """A positive integer at two significant figures, the way the frame writes it.

    Rounds half away from zero, not half to even. Both fourth and fifth rows
    land exactly on a tie (1.25 and 6.25), where Python's round() gives 1.2 and
    6.2 while the frame shows 1.3 and 6.3. The frame uses the convention every
    student was taught, so the frame is right and this follows it; using round()
    here would have reported the deck wrong over a floating-point default.

    Decimal on the exact integer, so the tie is a real tie rather than whatever
    the binary representation of 1.25e14 happens to be.
    """
    exp = len(str(x)) - 1
    mant = (Decimal(x) / Decimal(10) ** exp).quantize(Decimal("0.1"),
                                                      rounding=ROUND_HALF_UP)
    #% Trailing ".0" is written as an integer, matching the frame's "5" for the
    #% bigram row rather than "5.0".
    return "%s \\times 10^{%d}" % (mant.normalize(), exp)

print('torch', torch.__version__)
print(MODEL_CH01, 'at revision', REVISION_CH01)

torch 2.13.0+cpu
gpt2 at revision 607a30d783dfa663caf39e06633721c8d4cfcd7e


### Claim 1.2, traced: one forward pass returns a distribution, not a word

In [5]:
def claim_1_2_distribution(quiet=False):
    """One GPT-2 forward pass, and what actually comes back.

    ch01's claim 1.2 says a forward pass returns a distribution over the whole
    vocabulary rather than a word. Its instance has promised
    "notebook: ch01, GPT-2 over 50,257 tokens for one prompt" since the deck
    was authored, and resolved to nothing the entire time, because
    check_claims.py only tested for a source when the warrant was `measured`
    and 1.2 is `traced`. This is the function that makes the promise true, and
    the notebook lifts it verbatim rather than retyping it.

    What is checked, and what deliberately is not. The slide hedges the
    ordering: "with the exact ordering depending on the model version". So this
    does not pin the top word. Pinning it would make the artefact fail on a
    model update for a reason that has nothing to do with the claim, which is
    the definition of a brittle check. What it pins is what the claim actually
    asserts: the vocabulary is 50,257 wide, the numbers sum to one, and no
    single token owns the distribution.

    The first run of this function is why the slide now reads floor, bed,
    couch, ground, edge. It used to name `mat` and `table`, and the hedge above
    is about ordering, so tools/probe_cat_ranks.py went and asked where those
    two actually sit: `table` at rank 7 and `mat` at rank 31, p = 0.004. Rank 7
    the hedge carries; rank 31 is a false membership claim, and it had stood
    since the deck was authored because no gate in this repo compares a
    sentence of prose against a number. Plan step 9c, and the reason the
    ordering stays unpinned here is the same reason the membership could not
    stay unmeasured: the version-dependent part is the order, not the set's
    rough neighbourhood.

    mass_outside_top5 is here for teaching rather than for the gate. A student
    who has just been told "not a word, but 50,257 numbers" should see how much
    probability lives outside the five candidates the slide names.

    Returns None if the model will not load, the same contract attention_map
    has, so a committed artefact survives a machine with no network.
    """
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
    except Exception as e:
        if not quiet:
            print("  claim 1.2 skipped: transformers will not import (%r)" % (e,))
        return None
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_CH01,
                                            revision=REVISION_CH01)
        model = AutoModelForCausalLM.from_pretrained(MODEL_CH01,
                                                     revision=REVISION_CH01)
    except Exception as e:
        if not quiet:
            print("  claim 1.2 skipped: %s will not load (%r)" % (MODEL_CH01, e))
        return None
    model.eval()

    inputs = tok(PROMPT_CH01, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    #% The last position only. Every earlier position also carries a
    #% distribution, and saying so is the point of the frame after this one.
    probs = torch.softmax(logits[0, -1, :], dim=0)
    top = torch.topk(probs, 5)

    vocab = int(probs.shape[0])
    total = float(probs.sum())
    top5 = [{"token": tok.decode(int(i)), "prob": round(float(p), 4)}
            for p, i in zip(top.values, top.indices)]
    return {
        "model": MODEL_CH01,
        "revision": REVISION_CH01,
        "prompt": PROMPT_CH01,
        "n_input_tokens": int(inputs["input_ids"].shape[1]),
        "vocabulary_size": vocab,
        "vocabulary_size_tex": tex_int(vocab),
        "probabilities_sum": round(total, 6),
        "top5": top5,
        "top1_share": round(float(top.values[0]), 4),
        "mass_outside_top5": round(1 - float(top.values.sum()), 4),
        "_ok": (vocab == 50257 and abs(total - 1) < 1e-4
                and float(top.values[0]) < 0.9),
    }

r = claim_1_2_distribution()
r.pop('_ok', None)

print('model            ', r['model'], 'rev', r['revision'][:12])
print('prompt           ', repr(r['prompt']))
print('input tokens     ', r['n_input_tokens'])
print('vocabulary       ', r['vocabulary_size'], 'tokens')
print('probabilities    sum to', r['probabilities_sum'])
print()
print('top five of', r['vocabulary_size'], 'candidates:')
for t in r['top5']:
    print('   %-12s %.4f' % (repr(t['token']), t['prob']))
print()
print('the single best token holds', r['top1_share'])
print('everything outside the top five holds', r['mass_outside_top5'])

model             gpt2 rev 607a30d783df
prompt            'The cat sat on the'
input tokens      5
vocabulary        50257 tokens
probabilities    sum to 1.00001

top five of 50257 candidates:
   ' floor'     0.0764
   ' bed'       0.0653
   ' couch'     0.0541
   ' ground'    0.0521
   ' edge'      0.0478

the single best token holds 0.0764
everything outside the top five holds 0.7044


### Claim 1.6, derived: widening the context cannot buy you context

In [6]:
def claim_1_6_contexts():
    """The V-to-the-n table on ch01's Bengio frame, evaluated rather than typed.

    Claim 1.6 says an n-gram table's distinct contexts grow as the vocabulary
    raised to the context length, so context cannot be bought by widening the
    window. Its instance is "V to the n, evaluated on the frame at realistic
    V", and until now the four numbers in that tabular were typed by hand and
    checked by nobody.

    Needs no model and no network, unlike claim 1.2, so it runs even under
    --no-model. That asymmetry is worth stating: this claim is `derived` and
    arithmetic is the whole of its warrant, which is exactly why hand-typing
    the result was the wrong way to carry it.

    Also checks the sentence under the table, "a trillion-word corpus cannot
    populate the fourth row", because it is the line that makes the table an
    argument instead of a curiosity.

    _ok holds the arithmetic invariants only. Whether the frame agrees is not
    checked here but by deck_numbers, which looks the rendered strings up in
    ch01/deck.tex, so no copy of the table exists in this file to drift.
    """
    rows = []
    for name, n in (("bigram", 1), ("trigram", 2), ("4-gram", 3), ("5-gram", 4)):
        contexts = V_CH01 ** n
        rows.append({
            "model": name,
            "power": n,
            "contexts": contexts,
            "tex": _sci_tex(contexts),
        })
    fourth = rows[-1]["contexts"]
    return {
        "vocabulary": V_CH01,
        "rows": rows,
        "corpus_words": CORPUS_CH01,
        "fourth_row_over_corpus": fourth // CORPUS_CH01,
        "_ok": (rows[0]["contexts"] == V_CH01
                and rows[1]["contexts"] == V_CH01 ** 2
                and fourth > CORPUS_CH01),
    }

r = claim_1_6_contexts()
r.pop('_ok', None)

print('vocabulary V =', r['vocabulary'])
print()
for w in r['rows']:
    print('   %-8s V^%d %28d' % (w['model'], w['power'], w['contexts']))
print()
print('a corpus of', r['corpus_words'], 'words is short of the last')
print('row by a factor of', r['fourth_row_over_corpus'])

vocabulary V = 50000

   bigram   V^1                        50000
   trigram  V^2                   2500000000
   4-gram   V^3              125000000000000
   5-gram   V^4          6250000000000000000

a corpus of 1000000000000 words is short of the last
row by a factor of 6250000
